# Level 1 — Task 1: Data Preprocessing for Machine Learning
**Dataset:** Iris | **Tools:** pandas, scikit-learn, matplotlib, seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="muted")


## 2. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload iris.csv

df = pd.read_csv("iris.csv")
print(f"Shape: {df.shape}")
df.head()


## 3. Exploratory Data Analysis

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df["species"].value_counts()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

for ax, feat in zip(axes.flatten(), features):
    sns.histplot(data=df, x=feat, hue="species", kde=True, ax=ax, bins=20)
    ax.set_title(feat.replace("_", " ").title())

plt.suptitle("Feature Distributions by Species", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop(columns="species").corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()


## 4. Handle Missing Values

In [ ]:
# Check actual missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


In [ ]:
# Inject synthetic missing values to demonstrate imputation pipeline
rng = np.random.default_rng(42)
df_proc = df.copy()

missing_config = {"sepal_length": 8, "petal_width": 7}
for col, n in missing_config.items():
    idx = rng.choice(df_proc.index, size=n, replace=False)
    df_proc.loc[idx, col] = np.nan

print("Injected missing values:")
print(df_proc.isnull().sum())
print(f"\nTotal: {df_proc.isnull().sum().sum()} missing entries")


In [ ]:
# Visualize missing pattern before imputation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

missing_counts = df_proc.isnull().sum()
missing_counts[missing_counts > 0].plot(kind="bar", ax=axes[0], color="#E57373", edgecolor="white")
axes[0].set_title("Missing Values per Column")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

for col in ["sepal_length", "petal_width"]:
    axes[1].scatter(
        df_proc.index,
        df_proc[col],
        label=col,
        alpha=0.5,
        s=20
    )
    missing_idx = df_proc[df_proc[col].isna()].index
    axes[1].scatter(missing_idx, [df_proc[col].median()] * len(missing_idx),
                    marker="x", s=80, label=f"{col} (missing)", zorder=5)

axes[1].set_title("Missing Value Positions")
axes[1].set_xlabel("Index")
axes[1].legend(fontsize=8)
plt.suptitle("Missing Value Analysis", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Impute using median — no inplace to avoid pandas FutureWarning
impute_stats = {}
for col in df_proc.select_dtypes(include=np.number).columns:
    if df_proc[col].isnull().any():
        median_val = df_proc[col].median()
        df_proc[col] = df_proc[col].fillna(median_val)
        impute_stats[col] = median_val

print("Imputation applied (median):")
for col, val in impute_stats.items():
    print(f"  {col}: filled with {val:.4f}")

print(f"\nMissing values remaining: {df_proc.isnull().sum().sum()}")


In [ ]:
# Confirm with visual — distribution should not shift significantly after median imputation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for col, ax, title in zip(
    ["sepal_length", "petal_width"],
    axes,
    ["sepal_length", "petal_width"]
):
    sns.kdeplot(df[col], ax=ax, label="Original", linewidth=2)
    sns.kdeplot(df_proc[col], ax=ax, label="After Imputation", linewidth=2, linestyle="--")
    ax.axvline(df[col].median(), color="gray", linestyle=":", linewidth=1.2, label="Median")
    ax.set_title(f"{title} — Distribution Shift Check")
    ax.legend()

plt.suptitle("Distribution Before vs After Median Imputation", fontweight="bold")
plt.tight_layout()
plt.show()

print("\nMedian imputation preserves the original distribution shape.")
print("df_proc is clean and ready for encoding and scaling.")


## 5. Encode Categorical Variables

In [ ]:
# Label Encoding
le = LabelEncoder()
df_proc["species_encoded"] = le.fit_transform(df_proc["species"])
print("Label Encoding mapping:")
for cls, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} -> {enc}")


In [ ]:
# One-Hot Encoding
df_ohe = pd.get_dummies(df_proc["species"], prefix="species")
print("One-Hot Encoding sample:")
df_ohe.head()


## 6. Feature Scaling

In [ ]:
X = df_proc[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df_proc["species_encoded"]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Before scaling:")
print(X.describe().loc[["mean", "std"]].round(3))
print("\nAfter StandardScaler:")
print(X_scaled.describe().loc[["mean", "std"]].round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

X.plot(kind="box", ax=axes[0], patch_artist=True)
axes[0].set_title("Before Scaling")

X_scaled.plot(kind="box", ax=axes[1], patch_artist=True)
axes[1].set_title("After StandardScaler")

plt.suptitle("Feature Distribution: Before vs After Scaling", fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X_scaled)*100:.0f}%)")
print(f"Testing set  : {X_test.shape[0]} samples  ({X_test.shape[0]/len(X_scaled)*100:.0f}%)")
print(f"Features     : {X_train.shape[1]}")
print("\nClass distribution (train):")
print(pd.Series(y_train).map(dict(enumerate(le.classes_))).value_counts())


## 8. Summary

| Step | Action | Detail |
|---|---|---|
| Load | Read CSV | 150 rows, 5 columns |
| EDA | Distribution + Correlation | Visualized |
| Missing Values | Median imputation | Numerical columns |
| Encoding | Label + One-Hot | `species` column |
| Scaling | StandardScaler | Zero mean, unit variance |
| Split | 80/20 stratified | `random_state=42` |

`X_train`, `X_test`, `y_train`, `y_test` are ready for model training.